# Bài 51: Bộ tự mã hóa có che

Bộ tự mã hóa ở Bài 48 nén toàn bộ ảnh qua một vector thắt cổ chai hẹp. **Bộ tự mã hóa có che (masked autoencoders)** (MAE, <a href="../references.html#he-2021-mae">He et al., 2021</a>) tạo ra nút thắt cổ chai thông tin theo một cách hoàn toàn khác: cắt ảnh thành các mảng (phép chia mảng ViT ở Bài 47), che đi phần lớn chúng hoàn toàn — không làm nhiễu, không làm mờ, chỉ đơn giản là không bao giờ cho bộ mã hóa thấy — và huấn luyện mạng tái tạo các mảng còn thiếu từ bất cứ thứ gì còn lại. Đây là mô hình tiền huấn luyện tự giám sát thứ ba trong phần này, sau học tương phản (Bài 49) và tự chưng cất (Bài 50): không nhãn, không cặp âm, không mạng giáo viên — chỉ là một nhiệm vụ điền-vào-chỗ-trống nghe có vẻ hợp lý được tạo ra trực tiếp từ các ảnh không nhãn.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Chia mảng và che ngẫu nhiên

Cùng ảnh `16x16`, các mảng `4x4` như Bài 47 — tổng cộng 16 mảng. Với mỗi ảnh, chọn một tập con ngẫu nhiên để giữ **nhìn thấy được** và loại bỏ phần còn lại; các mảng bị loại bỏ không thuộc đầu vào của bộ mã hóa dưới bất kỳ hình thức nào, kể cả dưới dạng số 0 hay nhiễu.

In [ ]:
IMG = 16
PATCH = 4
N_PATCHES = (IMG // PATCH) ** 2  # 16
PATCH_DIM = PATCH * PATCH        # 16 (grayscale)

def patchify(img, patch_size=PATCH):
    B, C, H, W = img.shape
    P = patch_size
    patches = img.unfold(2, P, P).unfold(3, P, P)
    patches = patches.contiguous().view(B, C, -1, P, P).permute(0, 2, 1, 3, 4)
    return patches.reshape(B, -1, C * P * P)

def unpatchify(patches, patch_size=PATCH, img_size=IMG):
    B, N, D = patches.shape
    P = patch_size
    n_side = img_size // P
    x = patches.view(B, n_side, n_side, P, P).permute(0, 1, 3, 2, 4).contiguous()
    return x.view(B, 1, img_size, img_size)

def positional_encoding(T, D):
    pos = torch.arange(T).unsqueeze(1).float()
    i = torch.arange(D).unsqueeze(0).float()
    angle_rates = 1.0 / (10000 ** (2 * (i // 2) / D))
    angles = pos * angle_rates
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(angles[:, 0::2])
    pe[:, 1::2] = torch.cos(angles[:, 1::2])
    return pe

def random_masking(x, mask_ratio, generator=None):
    B, N, D = x.shape
    n_visible = max(1, int(N * (1 - mask_ratio)))
    noise = torch.rand(B, N, generator=generator)
    ids_shuffle = torch.argsort(noise, dim=1)      # random permutation of patch indices per image
    ids_keep = ids_shuffle[:, :n_visible]           # kept as VISIBLE, in original patch-index units
    ids_mask = ids_shuffle[:, n_visible:]            # HIDDEN entirely from the encoder
    x_visible = torch.gather(x, 1, ids_keep.unsqueeze(-1).expand(-1, -1, D))
    return x_visible, ids_keep, ids_mask

def make_image(shape_type, cx, cy, size=IMG):
    img = np.zeros((size, size), dtype=np.float32)
    if shape_type == 'plus':
        img[cy-1:cy+2, cx-3:cx+4] = 1.0
        img[cy-3:cy+4, cx-1:cx+2] = 1.0
    else:
        yy, xx = np.mgrid[0:size, 0:size]
        img[((xx-cx)**2 + (yy-cy)**2) <= 9] = 1.0
    return img

def make_dataset(rng_local, n, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        imgs.append(make_image(shape_type, cx, cy))
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

rng = np.random.default_rng(6)
X_train, y_train = make_dataset(rng, 400)
X_test, y_test = make_dataset(rng, 150)
Xt = torch.tensor(X_train).unsqueeze(1)
Xte = torch.tensor(X_test).unsqueeze(1)

print(f'{N_PATCHES} patches per image, {PATCH_DIM} pixels each')

## Một bộ mã hóa-giải mã bất đối xứng

**Bộ mã hóa** là một Transformer nhỏ (Bài 46) chỉ luôn xử lý các mảng nhìn thấy được — các mảng bị che không bao giờ đi vào nó, nên bộ mã hóa nhanh và không bao giờ lãng phí tính toán vào các token sẽ bị vứt bỏ. **Bộ giải mã** là nơi các mảng bị che xuất hiện trở lại: một vector **token che (mask token)** dùng chung, học được đại diện cho mọi mảng bị che, kết hợp với mã hóa vị trí gốc của mảng đó (để bộ giải mã biết mỗi chỗ trống thuộc về *đâu*, dù nó chưa bao giờ thấy *thứ gì* ở đó), và một Transformer gọn nhẹ dự đoán giá trị điểm ảnh thô của mỗi mảng bị che. Hàm mất mát **chỉ được tính trên các mảng bị che** — mạng không được tính công cho việc sao chép các điểm ảnh mà nó đã được cho thấy.

In [ ]:
class MAEEncoder(nn.Module):
    def __init__(self, patch_dim=PATCH_DIM, embed_dim=32, n_heads=4, n_layers=2):
        super().__init__()
        self.embed = nn.Linear(patch_dim, embed_dim)
        self.register_buffer('pe', positional_encoding(N_PATCHES, embed_dim))
        layer = nn.TransformerEncoderLayer(embed_dim, n_heads, dim_feedforward=embed_dim * 2,
                                            batch_first=True, dropout=0.0)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

    def forward(self, x_visible, ids_keep):
        tok = self.embed(x_visible) + self.pe[ids_keep]  # positional encoding at each patch's TRUE index
        return self.encoder(tok)

class MAEDecoder(nn.Module):
    def __init__(self, embed_dim=32, decoder_dim=16, patch_dim=PATCH_DIM, n_heads=4, n_layers=1):
        super().__init__()
        self.embed = nn.Linear(embed_dim, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        self.register_buffer('pe', positional_encoding(N_PATCHES, decoder_dim))
        layer = nn.TransformerEncoderLayer(decoder_dim, n_heads, dim_feedforward=decoder_dim * 2,
                                            batch_first=True, dropout=0.0)
        self.decoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.pred = nn.Linear(decoder_dim, patch_dim)
        self.decoder_dim = decoder_dim

    def forward(self, encoded_visible, ids_keep, ids_mask):
        B, n_mask = ids_mask.shape
        vis_tok = self.embed(encoded_visible) + self.pe[ids_keep]
        mask_tok = self.mask_token.expand(B, n_mask, -1) + self.pe[ids_mask]
        full = torch.zeros(B, N_PATCHES, self.decoder_dim)
        full = full.scatter(1, ids_keep.unsqueeze(-1).expand(-1, -1, self.decoder_dim), vis_tok)
        full = full.scatter(1, ids_mask.unsqueeze(-1).expand(-1, -1, self.decoder_dim), mask_tok)
        return self.pred(self.decoder(full))  # (B, N_PATCHES, patch_dim), full sequence, original order

class MAE(nn.Module):
    def __init__(self, mask_ratio=0.5, embed_dim=32, decoder_dim=16):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.encoder = MAEEncoder(embed_dim=embed_dim)
        self.decoder = MAEDecoder(embed_dim=embed_dim, decoder_dim=decoder_dim)

    def forward(self, img, generator=None):
        patches = patchify(img)
        x_visible, ids_keep, ids_mask = random_masking(patches, self.mask_ratio, generator=generator)
        encoded = self.encoder(x_visible, ids_keep)
        pred = self.decoder(encoded, ids_keep, ids_mask)
        D = patches.shape[-1]
        pred_masked = torch.gather(pred, 1, ids_mask.unsqueeze(-1).expand(-1, -1, D))
        target_masked = torch.gather(patches, 1, ids_mask.unsqueeze(-1).expand(-1, -1, D))
        loss = F.mse_loss(pred_masked, target_masked)
        return loss, pred, ids_mask, ids_keep, patches

def train_mae(mask_ratio, epochs=600, lr=0.001, seed=0):
    torch.manual_seed(seed)
    model = MAE(mask_ratio=mask_ratio)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss, *_ = model(Xt)
        loss.backward()
        opt.step()
    return model

## Có thể tái tạo được bao nhiêu từ ít đến vậy?

Quét tỉ lệ mảng bị che từ 25% đến 75%, và so sánh sai số tái tạo mảng bị che của mỗi mô hình với một đường cơ sở tầm thường: dự đoán giá trị điểm ảnh trung bình của tập huấn luyện cho mọi mảng bị che, bất kể ảnh nào.

In [ ]:
train_patches = patchify(Xt)
mean_pixel = train_patches.mean()

print(f'{"mask ratio":>10} {"MAE test MSE":>14} {"mean-pixel baseline":>20}')
mae_models = {}
for mr in [0.25, 0.5, 0.75]:
    model = train_mae(mask_ratio=mr)
    mae_models[mr] = model
    with torch.no_grad():
        test_loss, pred, ids_mask, ids_keep, patches = model(Xte, generator=torch.Generator().manual_seed(123))
        D = patches.shape[-1]
        target_masked = torch.gather(patches, 1, ids_mask.unsqueeze(-1).expand(-1, -1, D))
        baseline_mse = F.mse_loss(torch.full_like(target_masked, mean_pixel), target_masked).item()
    print(f'{mr:>10} {test_loss.item():>14.4f} {baseline_mse:>20.4f}')

Việc tái tạo trở nên khó hơn khi nhiều phần ảnh bị che hơn — không ngạc nhiên, đơn giản là có ít bằng chứng hơn để làm việc — nhưng mô hình vượt đường cơ sở ngây thơ một khoảng lớn ngay cả ở mức che 75%, nơi chỉ 4 trong 16 mảng từng được cho bộ mã hóa thấy. Điều đó chỉ khả thi vì các hình này có sự dư thừa nội tại rất lớn: hình bóng của một hình tròn được xác định hoàn toàn bởi tâm và bán kính của nó, nên một nhúm mảng biên nhìn thấy được, kết hợp với cơ chế chú ý trên tất cả chúng, là đủ để suy ra phần còn lại. MAE thực tế phát hiện cùng điều đó ở quy mô lớn hơn nhiều — ảnh chụp tự nhiên cũng có tính dư thừa không gian cao (một mảng trời, một mảng da, một mảng gạch dự đoán được các lân cận của nó) — đó là *lý do* He và cộng sự phát hiện ra tỉ lệ che tiền huấn luyện tốt nhất cho ảnh là khoảng 75%, cao hơn hẳn tỉ lệ che từ ~15% mà BERT dùng cho ngôn ngữ, nơi phần dự đoán được của một câu từ phần còn lại ít hơn nhiều.

In [ ]:
idx = 7
model75 = mae_models[0.75]
with torch.no_grad():
    _, pred, ids_mask, ids_keep, patches = model75(Xte, generator=torch.Generator().manual_seed(42))

masked_view = patches.clone()
masked_view[torch.arange(len(patches)).unsqueeze(1), ids_mask] = 0.5  # gray out hidden patches
recon_view = patches.clone()
recon_view[torch.arange(len(patches)).unsqueeze(1), ids_mask] = pred[torch.arange(len(patches)).unsqueeze(1), ids_mask]

orig_img = unpatchify(patches)[idx, 0].numpy()
masked_img = unpatchify(masked_view)[idx, 0].numpy()
recon_img = unpatchify(recon_view)[idx, 0].numpy()

fig, axes = plt.subplots(1, 3, figsize=(7, 2.8))
for ax, im, title in zip(axes, [orig_img, masked_img, recon_img],
                          ['original', 'what the encoder saw\n(75% hidden)', 'reconstructed']):
    ax.imshow(im, cmap='gray', vmin=0, vmax=1); ax.set_title(title, fontsize=9); ax.axis('off')
plt.show()

## Từ tái tạo đến biểu diễn: một đầu dò tuyến tính

Bản thân nhiệm vụ tái tạo bị vứt bỏ sau tiền huấn luyện — thứ MAE thực tế giữ lại là **bộ mã hóa**, đúng như Bài 48 và 49 chỉ giữ lại bộ mã hóa/giáo viên đã huấn luyện và loại bỏ bộ máy tương phản/chưng cất xung quanh nó. Đóng băng bộ mã hóa tỉ-lệ-che-0.75, đưa vào *mọi* mảng cùng lúc (không che, `mask_ratio=0` lúc đánh giá — toàn bộ mục đích của việc tiền huấn luyện dưới một nút thắt cổ chai khắc nghiệt là để chuẩn bị bộ mã hóa cho trường hợp dễ), gộp trung bình đầu ra của nó thành một vector đặc trưng cho mỗi ảnh, và huấn luyện một đầu dò tuyến tính trên một nhúm mẫu có nhãn.

In [ ]:
def encode_full(encoder, img):
    patches = patchify(img)
    B, N, _ = patches.shape
    ids_all = torch.arange(N).unsqueeze(0).expand(B, -1)
    with torch.no_grad():
        encoded = encoder(patches, ids_all)
    return encoded.mean(dim=1)  # average-pool over all 16 patches -> one vector per image

def make_noisy_dataset(rng_local, n, noise=0.2, position_range=(4, 12)):
    imgs, labels = [], []
    for _ in range(n):
        shape_type = rng_local.choice(['plus', 'circle'])
        cx, cy = rng_local.integers(*position_range), rng_local.integers(*position_range)
        img = make_image(shape_type, cx, cy)
        img = np.clip(img + rng_local.normal(0, noise, img.shape), 0, 1).astype(np.float32)
        imgs.append(img)
        labels.append(0 if shape_type == 'plus' else 1)
    return np.array(imgs, dtype=np.float32), np.array(labels, dtype=np.int64)

probe_rng = np.random.default_rng(11)
X_probe_train, y_probe_train = make_noisy_dataset(probe_rng, 8)
X_probe_test, y_probe_test = make_noisy_dataset(probe_rng, 150)
Xpt = torch.tensor(X_probe_train).unsqueeze(1)
Xpte = torch.tensor(X_probe_test).unsqueeze(1)

def linear_probe_acc(encoder, seed):
    torch.manual_seed(seed)
    feat_train = encode_full(encoder, Xpt)
    feat_test = encode_full(encoder, Xpte)
    probe = nn.Linear(feat_train.shape[1], 2)
    opt = torch.optim.Adam(probe.parameters(), lr=0.05)
    ytr = torch.tensor(y_probe_train)
    for _ in range(300):
        opt.zero_grad()
        loss = F.cross_entropy(probe(feat_train), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        preds = probe(feat_test).argmax(1).numpy()
    return (preds == y_probe_test).mean()

mae_accs, random_accs = [], []
for seed in range(5):
    mae_accs.append(linear_probe_acc(mae_models[0.75].encoder, seed=seed + 50))
    torch.manual_seed(seed)
    random_encoder = MAEEncoder()
    random_accs.append(linear_probe_acc(random_encoder, seed=seed + 50))

print(f'linear probe on RANDOM (untrained) MAE encoder: {np.mean(random_accs):.1%}  (+/- {np.std(random_accs):.1%})')
print(f'linear probe on MAE-pretrained encoder:         {np.mean(mae_accs):.1%}  (+/- {np.std(mae_accs):.1%})')

Đó là một con số thực sự đáng thất vọng — các đặc trưng đóng băng, gộp trung bình của bộ mã hóa tiền huấn luyện bằng MAE cho kết quả dò *tệ hơn* so với của một bộ mã hóa ngẫu nhiên, chưa huấn luyện. Đây không phải lỗi; đó là một thuộc tính thực sự, đã được ghi nhận của tiền huấn luyện tái tạo có che, và đáng để hiểu tại sao. Học tương phản (Bài 49) và tự chưng cất (Bài 50) đều trực tiếp tối ưu một vector nhúng toàn ảnh đã gộp để tự nó hữu ích — đó là toàn bộ tín hiệu huấn luyện. Tín hiệu huấn luyện của MAE hoàn toàn không chạm tới một biểu diễn đã gộp: bộ mã hóa chỉ phải tạo ra các token theo từng mảng đủ phong phú để cơ chế chú ý *bên trong bộ giải mã* có thể điền các mảng còn thiếu. Không có gì buộc *trung bình* của các token đó phải là một bản tóm tắt tốt về "đây là hình gì" — một biểu diễn hoàn toàn tốt cho việc hỗ trợ tái tạo có thể khi lấy trung bình lại thành thứ mà một đầu dò tuyến tính thấy vô dụng. Bài báo MAE gốc báo cáo chính xác mẫu hình này ở quy mô thực: độ chính xác đầu dò tuyến tính thấp hơn đáng kể so với các phương pháp tương phản, với thành quả thực sự của MAE chỉ thể hiện sau **tinh chỉnh toàn phần (full fine-tuning)** (cập nhật mọi trọng số của bộ mã hóa trên nhiệm vụ hạ nguồn, không chỉ một phần đầu tuyến tính bên trên các đặc trưng đóng băng) — một câu chuyện triển khai thực sự khác với Bài 48 và 49, chứ không phải tệ hơn.

## Ba mô hình tự giám sát, một mẫu hình chung

Học tương phản (Bài 49), tự chưng cất (Bài 50) và tái tạo có che (bài này) tạo ra ba tín hiệu huấn luyện hoàn toàn khác nhau từ cùng một nguyên liệu thô — các ảnh không nhãn — vậy mà cả ba đều tồn tại để trả lời cùng một câu hỏi: nhiệm vụ phá hủy thông tin rẻ nhất nào mà lời giải của nó buộc một mạng phải học được thứ gì đó thực sự hữu ích về ảnh? Học tương phản phá hủy danh tính qua các góc nhìn tăng cường và yêu cầu mạng khôi phục nó; tự chưng cất phá hủy sự chắc chắn của giáo viên và yêu cầu học sinh vẫn khớp với nó; tái tạo có che phá hủy thẳng phần lớn ảnh và yêu cầu mạng điền phần còn lại. Nhưng kết quả đầu dò của bài này là một lời cảnh báo thực sự chống lại việc giả định cả ba tạo ra các biểu diễn có thể thay thế cho nhau: tín hiệu huấn luyện của MAE rẻ về mặt kiến trúc (bộ mã hóa thậm chí không bao giờ xử lý các token bị che, khác với hai phương pháp kia, vốn luôn thấy toàn bộ ảnh) và xuất sắc cho *tinh chỉnh* hạ nguồn, nhưng nó có được hiệu quả đó nhờ không bao giờ buộc một vector tóm tắt đã gộp, hữu ích tuyến tính phải tồn tại — chính là thứ mà học tương phản và tự chưng cất được xây dựng xoay quanh. Lựa chọn giữa ba phương pháp này trong thực tế là một sự đánh đổi kỹ thuật thực sự, không phải một bảng xếp hạng nghiêm ngặt.

### Bài tập

1. Tăng `mask_ratio` lên `0.9` (chỉ 1-2 mảng nhìn thấy được). MSE tái tạo có còn vượt đường cơ sở điểm ảnh trung bình không, hay nó sụp đổ về gần bằng sai số của đường cơ sở — và điều đó có tiết lộ một ngưỡng mà vượt qua thì sự dư thừa của tập dữ liệu này cạn kiệt không?
2. Bộ giải mã ở đây được cố tình làm nhỏ xíu (`n_layers=1`, `decoder_dim=16`, nhỏ hơn bộ mã hóa). Hãy thử làm bộ giải mã *lớn hơn* bộ mã hóa (ví dụ `decoder_dim=64`, `n_layers=3`) trong khi giữ bộ mã hóa cố định. Chất lượng tái tạo có cải thiện nhiều không — và điều đó có khớp với lựa chọn thiết kế bộ giải mã nhỏ của MAE thực tế, với lập luận rằng dù sao bộ giải mã cũng bị loại bỏ sau tiền huấn luyện không?
3. Thay vì đóng băng bộ mã hóa cho đầu dò tuyến tính, hãy tinh chỉnh nó từ đầu đến cuối: để gradient từ hàm mất mát phân loại 8 mẫu cập nhật cả trọng số của chính bộ mã hóa, không chỉ một phần đầu tuyến tính bên trên các đặc trưng đóng băng. Tinh chỉnh đầu-cuối có lấy lại được khoảng cách giữa MAE và đường cơ sở ngẫu nhiên không — khớp với phát hiện của bài báo MAE thực tế rằng các đặc trưng MAE đã tinh chỉnh rất xuất sắc, ngay cả khi các đặc trưng đóng băng cho kết quả dò kém?